# E2 — MySQL Buffer Pool + Cross-Layer Trap Analysis

This notebook covers two closely related experiments:

**E2-mysql-buffer** — InnoDB buffer pool undersizing (L3 knob: `MYSQL_BUFFER_POOL`)  
&emsp;S0-baseline vs S4-mysql-buffer-128m × W3-grade × {low, medium, high}

**E2b-cross-layer-trap** — Same user symptom, different root cause  
&emsp;S5-mysql-conns-30 vs S6-quarkus-pool-5 × W3-grade × {medium, high}

## Research question

1. **E2:** Does the buffer pool miss rate (>5%) correctly identify L3-MySQL as the bottleneck?
2. **E2b (THE KEY FIGURE):** When S5 and S6 produce the same user-facing p95 spike,  
   do the diagnostic signals correctly *disambiguate* them?
   - S5: `threads_connected / max_connections ≈ 1.0` (DB ceiling hit)
   - S6: `agroal_awaiting_count > 0` while `threads_running ≪ max_connections` (app pool hit)

## Figures produced
- **Fig 1** — Buffer miss rate time series (S0 vs S4)
- **Fig 2** — p95 latency and throughput comparison (S0 vs S4)
- **Fig 3** — **Cross-layer trap**: user symptom (p95) looks identical for S5/S6;  
  diagnostic signals are clearly different (the paper's key contribution)

Run `experiments/E2-mysql-buffer/run.py` first.

In [ ]:
import json
import csv
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yaml
import pandas as pd
from IPython.display import display

matplotlib.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.fontsize': 10, 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

REPO_ROOT = Path('.').resolve().parent.parent
RESULTS_DIR = REPO_ROOT / 'results'
FIGURES_DIR = Path('.') / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

E2_KEY  = 'E2-mysql-buffer'
E2B_KEY = 'E2b-cross-layer-trap'

COLORS = {
    'S0-baseline':          '#2196F3',
    'S4-mysql-buffer-128m': '#F44336',
    'S5-mysql-conns-30':    '#FF9800',
    'S6-quarkus-pool-5':    '#9C27B0',
}
LABELS = {
    'S0-baseline':          'S0 baseline',
    'S4-mysql-buffer-128m': 'S4 buffer-128m (bad)',
    'S5-mysql-conns-30':    'S5 mysql-conns-30',
    'S6-quarkus-pool-5':    'S6 quarkus-pool-5',
}
INTENSITIES = ['low', 'medium', 'high']

In [ ]:
def load_locust_stats(run_dir: Path) -> dict:
    f = run_dir / 'locust_stats.csv'
    if not f.exists():
        return {}
    rows = {}
    with open(f) as fh:
        for row in csv.DictReader(fh):
            name = row.get('Name', '')
            try:
                total = float(row.get('Request Count', 1) or 1)
                rows[name] = {
                    'p95':        float(row.get('95%', 0) or 0),
                    'p99':        float(row.get('99%', 0) or 0),
                    'error_rate': float(row.get('Failure Count', 0) or 0) / max(1, total),
                    'rps':        float(row.get('Requests/s', 0) or 0),
                }
            except (ValueError, TypeError):
                pass
    return rows


def get_prom_series(prom: dict, key: str):
    data = prom.get(key, {})
    if data.get('status') != 'success':
        return [], []
    ts_list, v_list = [], []
    for series in data.get('data', {}).get('result', []):
        for ts, v in series.get('values', []):
            try:
                ts_list.append(float(ts))
                v_list.append(float(v))
            except (ValueError, TypeError):
                pass
    return ts_list, v_list


def load_run(run_id: str) -> dict:
    d = RESULTS_DIR / run_id
    meta = yaml.safe_load((d / 'metadata.yaml').read_text()) if (d / 'metadata.yaml').exists() else {}
    prom = json.loads((d / 'prom_snapshot.json').read_text()) if (d / 'prom_snapshot.json').exists() else {}
    return {'run_id': run_id, 'meta': meta, 'prom': prom, 'stats': load_locust_stats(d)}

In [ ]:
runs_e2, runs_e2b = [], []
if RESULTS_DIR.exists():
    for run_dir in sorted(RESULTS_DIR.iterdir()):
        mf = run_dir / 'metadata.yaml'
        if not mf.exists():
            continue
        meta = yaml.safe_load(mf.read_text())
        exp = meta.get('experiment', '')
        if exp == E2_KEY:
            runs_e2.append(load_run(run_dir.name))
        elif exp == E2B_KEY:
            runs_e2b.append(load_run(run_dir.name))

all_runs = runs_e2 + runs_e2b
print(f'E2-mysql-buffer:       {len(runs_e2)} run(s)')
print(f'E2b-cross-layer-trap:  {len(runs_e2b)} run(s)')
for r in all_runs:
    m = r['meta']
    print(f"  {r['run_id'][:8]}  exp={m.get('experiment','?')[:12]:12s}  "
          f"scenario={m.get('scenario','?')}  intensity={m.get('intensity','?')}")

if not all_runs:
    print('\nNo results yet. Generate data with:')
    print('  python experiments/E2-mysql-buffer/run.py --intensities medium')

In [ ]:
records = []
for r in all_runs:
    m, prom, stats = r['meta'], r['prom'], r['stats']
    agg = stats.get('Aggregated', next(iter(stats.values()), {}))

    _, reads     = get_prom_series(prom, 'mysql_global_status_innodb_buffer_pool_reads')
    _, read_reqs = get_prom_series(prom, 'mysql_global_status_innodb_buffer_pool_read_requests')
    miss_rates   = [r_ / rr for r_, rr in zip(reads, read_reqs) if rr > 0]

    _, connected = get_prom_series(prom, 'mysql_global_status_threads_connected')
    _, max_conns = get_prom_series(prom, 'mysql_global_variables_max_connections')
    conn_ratios  = [c / mx for c, mx in zip(connected, max_conns) if mx > 0]

    _, awaiting  = get_prom_series(prom, 'agroal_awaiting_count')
    _, db_threads = get_prom_series(prom, 'mysql_global_status_threads_running')

    records.append({
        'experiment':       m.get('experiment', '?'),
        'scenario':         m.get('scenario', '?'),
        'intensity':        m.get('intensity', '?'),
        'p95_ms':           agg.get('p95', 0),
        'p99_ms':           agg.get('p99', 0),
        'error_rate_%':     round(agg.get('error_rate', 0) * 100, 2),
        'rps':              agg.get('rps', 0),
        'miss_rate_mean':   round(float(np.mean(miss_rates)), 4)   if miss_rates  else 0,
        'conn_ratio_max':   round(max(conn_ratios), 3)             if conn_ratios else 0,
        'agroal_await_mean':round(float(np.mean(awaiting)), 2)     if awaiting    else 0,
        'db_threads_mean':  round(float(np.mean(db_threads)), 1)   if db_threads  else 0,
    })

df = pd.DataFrame(records)
if not df.empty:
    display(df.sort_values(['experiment', 'scenario', 'intensity']))
else:
    print('No data yet.')

In [ ]:
# Figure 1 — Buffer pool miss rate time series (S0 vs S4, medium intensity)

target = {'S0-baseline': None, 'S4-mysql-buffer-128m': None}
for r in runs_e2:
    sc = r['meta'].get('scenario')
    it = r['meta'].get('intensity')
    if it == 'medium' and sc in target:
        target[sc] = r

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Fig 1 — E2 MySQL Buffer Pool: Miss Rate and Query Latency (W3-grade, medium)',
             fontsize=12, fontweight='bold')

for scenario, r in target.items():
    c = COLORS[scenario]
    lbl = LABELS[scenario]

    if r is None:
        for ax in axes:
            ax.text(0.5, 0.5, f'Run for {scenario}\nnot found',
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=10, color='gray', style='italic')
        continue

    prom = r['prom']
    ts_r,   reads     = get_prom_series(prom, 'mysql_global_status_innodb_buffer_pool_reads')
    _,      read_reqs = get_prom_series(prom, 'mysql_global_status_innodb_buffer_pool_read_requests')

    if ts_r and reads and read_reqs:
        miss = [r_ / rr if rr > 0 else 0 for r_, rr in zip(reads, read_reqs)]
        t_rel = [(t - ts_r[0]) / 60 for t in ts_r]
        axes[0].plot(t_rel, miss, color=c, label=lbl, linewidth=2)

    ts_h, http_sum = get_prom_series(prom, 'http_server_requests_seconds_sum')
    _, http_cnt    = get_prom_series(prom, 'http_server_requests_seconds_count')
    if ts_h and http_sum and http_cnt:
        avg_ms = [s / c_ * 1000 if c_ > 0 else 0 for s, c_ in zip(http_sum, http_cnt)]
        t_rel  = [(t - ts_h[0]) / 60 for t in ts_h]
        axes[1].plot(t_rel, avg_ms, color=c, label=lbl, linewidth=2)

axes[0].axhline(0.05, color='orange', linestyle='--', linewidth=1.2, alpha=0.8,
                label='diagnostic threshold (5% miss)')
axes[0].set_title('InnoDB Buffer Pool Miss Rate')
axes[0].set_xlabel('Time into run (min)')
axes[0].set_ylabel('reads / read_requests')
axes[0].set_ylim(0, None)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Mean Request Latency (ms)')
axes[1].set_xlabel('Time into run (min)')
axes[1].set_ylabel('latency (ms)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'E2-fig1-buffer-miss-rate.pdf')
plt.show()
print('Saved: figures/E2-fig1-buffer-miss-rate.pdf')

In [ ]:
# Figure 2 — p95 latency comparison across intensities (S0 vs S4)

e2_df = df[df['experiment'] == E2_KEY] if not df.empty else pd.DataFrame()

if e2_df.empty:
    print('No E2 data for Figure 2.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Fig 2 — E2 SLO Impact: p95 Latency and Throughput (S0 vs S4)',
                 fontsize=12, fontweight='bold')

    x     = np.arange(len(INTENSITIES))
    width = 0.35
    scenarios = ['S0-baseline', 'S4-mysql-buffer-128m']

    for idx, scenario in enumerate(scenarios):
        sub = e2_df[e2_df['scenario'] == scenario].set_index('intensity')
        p95  = [sub.loc[i, 'p95_ms'] if i in sub.index else 0 for i in INTENSITIES]
        rps  = [sub.loc[i, 'rps']    if i in sub.index else 0 for i in INTENSITIES]
        off  = (idx - 0.5) * width
        axes[0].bar(x + off, p95, width, label=LABELS[scenario],
                    color=COLORS[scenario], alpha=0.85)
        axes[1].bar(x + off, rps, width, label=LABELS[scenario],
                    color=COLORS[scenario], alpha=0.85)

    for ax, title, ylabel in [
        (axes[0], 'p95 Latency (ms)', 'p95 latency (ms)'),
        (axes[1], 'Throughput (req/s)', 'requests/s'),
    ]:
        ax.set_title(title)
        ax.set_xlabel('Intensity')
        ax.set_ylabel(ylabel)
        ax.set_xticks(x)
        ax.set_xticklabels(INTENSITIES)
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'E2-fig2-slo-impact.pdf')
    plt.show()
    print('Saved: figures/E2-fig2-slo-impact.pdf')

In [ ]:
# Figure 3 — THE CROSS-LAYER TRAP (the paper's key figure)
#
# Top row:    user-visible symptom — p95 latency for S5 and S6 looks the same
# Middle row: diagnostic signal S5 — threads_connected / max_connections plateaus at 1.0
# Bottom row: diagnostic signal S6 — agroal_awaiting > 0, DB threads well below max
#
# WITHOUT the diagnostic signals, you cannot tell which layer to fix.

target_trap = {s: None for s in ['S5-mysql-conns-30', 'S6-quarkus-pool-5']}
for r in runs_e2b:
    sc = r['meta'].get('scenario')
    it = r['meta'].get('intensity')
    if it == 'high' and sc in target_trap:
        target_trap[sc] = r
    elif it == 'medium' and sc in target_trap and target_trap[sc] is None:
        target_trap[sc] = r

fig = plt.figure(figsize=(14, 12))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle(
    'Fig 3 — Cross-Layer Trap: S5 (MySQL max_connections) vs S6 (Quarkus pool)\n'
    'Same user symptom · Different root cause · Different fix',
    fontsize=13, fontweight='bold'
)

trap_scenarios = ['S5-mysql-conns-30', 'S6-quarkus-pool-5']
trap_titles    = ['S5: mysql-conns-30  →  fix: MYSQL_MAX_CONNS=200',
                  'S6: quarkus-pool-5  →  fix: QUARKUS_DB_POOL_MAX=20']

for col, (scenario, title) in enumerate(zip(trap_scenarios, trap_titles)):
    r = target_trap[scenario]
    c = COLORS[scenario]

    # --- Row 0: user symptom (p95 latency time series) ---
    ax0 = fig.add_subplot(gs[0, col])
    ax0.set_title(f'User symptom (p95 latency)\n{title}', fontsize=10)

    if r is None:
        ax0.text(0.5, 0.5, f'Run for {scenario}\nnot found',
                 transform=ax0.transAxes, ha='center', va='center',
                 fontsize=9, color='gray', style='italic')
    else:
        prom = r['prom']
        ts_h, http_sum = get_prom_series(prom, 'http_server_requests_seconds_sum')
        _, http_cnt    = get_prom_series(prom, 'http_server_requests_seconds_count')
        if ts_h and http_sum and http_cnt:
            avg_ms = [s / cnt * 1000 if cnt > 0 else 0 for s, cnt in zip(http_sum, http_cnt)]
            t_rel  = [(t - ts_h[0]) / 60 for t in ts_h]
            ax0.plot(t_rel, avg_ms, color=c, linewidth=2)
        ax0.set_xlabel('Time (min)')
        ax0.set_ylabel('avg latency (ms)')
        ax0.grid(True, alpha=0.3)

    # --- Row 1: S5 diagnostic — threads_connected / max_connections ---
    ax1 = fig.add_subplot(gs[1, col])
    if scenario == 'S5-mysql-conns-30':
        ax1.set_title('S5 diagnostic: threads_connected / max_connections', fontsize=10)
        if r is not None:
            prom = r['prom']
            ts_c, connected = get_prom_series(prom, 'mysql_global_status_threads_connected')
            _, max_c        = get_prom_series(prom, 'mysql_global_variables_max_connections')
            if ts_c and connected and max_c:
                ratios = [c_ / mx if mx > 0 else 0 for c_, mx in zip(connected, max_c)]
                t_rel  = [(t - ts_c[0]) / 60 for t in ts_c]
                ax1.plot(t_rel, ratios, color=c, linewidth=2, label='conn / max_conns')
            ax1.axhline(0.95, color='red', linestyle='--', alpha=0.7, label='plateau threshold (0.95)')
            ax1.set_ylim(0, 1.1)
            ax1.set_ylabel('ratio')
            ax1.legend(fontsize=9)
            ax1.grid(True, alpha=0.3)
    else:
        ax1.set_title('S5 diagnostic: N/A for this scenario', fontsize=10)
        ax1.text(0.5, 0.5, 'threads_connected / max_connections\ndoes NOT plateau for S6\n(DB has capacity)',
                 transform=ax1.transAxes, ha='center', va='center',
                 fontsize=9, color='green', fontweight='bold')
        ax1.set_facecolor('#f0fff0')
    ax1.set_xlabel('Time (min)')

    # --- Row 2: S6 diagnostic — agroal_awaiting_count ---
    ax2 = fig.add_subplot(gs[2, col])
    if scenario == 'S6-quarkus-pool-5':
        ax2.set_title('S6 diagnostic: agroal_awaiting_count and DB threads', fontsize=10)
        if r is not None:
            prom = r['prom']
            ts_a, await_ = get_prom_series(prom, 'agroal_awaiting_count')
            _, db_thr    = get_prom_series(prom, 'mysql_global_status_threads_running')
            if ts_a and await_:
                t_rel = [(t - ts_a[0]) / 60 for t in ts_a]
                ax2.plot(t_rel, await_, color=c, linewidth=2, label='agroal_awaiting (>0 = bottleneck)')
            if db_thr:
                ax2.plot(t_rel[:len(db_thr)], db_thr, color='gray', linewidth=1.5,
                         linestyle='--', label='DB threads_running (idle)')
            ax2.set_ylabel('count')
            ax2.legend(fontsize=9)
            ax2.grid(True, alpha=0.3)
    else:
        ax2.set_title('S6 diagnostic: N/A for this scenario', fontsize=10)
        ax2.text(0.5, 0.5, 'agroal_awaiting_count stays at 0\nfor S5 (app pool is not the limit)',
                 transform=ax2.transAxes, ha='center', va='center',
                 fontsize=9, color='green', fontweight='bold')
        ax2.set_facecolor('#f0fff0')
    ax2.set_xlabel('Time (min)')

fig.savefig(FIGURES_DIR / 'E2b-fig3-cross-layer-trap.pdf')
plt.show()
print('Saved: figures/E2b-fig3-cross-layer-trap.pdf')
print('\nThis is the paper\'s key figure: same user symptom, diagnostic signals are disjoint.')

## Interpretation

### E2 — Buffer pool
| Metric | S0 baseline | S4 buffer-128m |
|--------|-------------|----------------|
| miss_rate_mean | < 0.01 | > 0.05 |
| p95 latency | baseline | > 3× baseline |

### E2b — Cross-layer trap (the paper's core figure)

| | S5: mysql-conns-30 | S6: quarkus-pool-5 |
|--|--|--|
| **User symptom** | p95 high | p95 high (same!) |
| **S5 signal** | `threads_connected / max_connections ≈ 1.0` | `threads_connected ≪ max_connections` |
| **S6 signal** | `agroal_awaiting = 0` | `agroal_awaiting > 0` sustained |
| **Correct fix** | `MYSQL_MAX_CONNS=200` | `QUARKUS_DB_POOL_MAX=20` |
| **Wrong fix** | Setting pool to 20 doesn't help | Setting max_connections to 200 doesn't help |

**Claim:** A system that uses only SLO signals (p95, error rate) cannot distinguish S5 from S6.
The recommender must inspect the diagnostic signal layer to correctly identify the root cause.
This is the gap between existing approaches (CherryPick) and our evidence-grounded recommender.